인보응 프로젝트
1차 실험 결과

전반적으로 적응형 위험 토큰 사전 기반 URL 분류 구조가 정상적으로 동작했으나, phishing 클래스의 F1-score가 0.76으로 다른 클래스에 비해 낮게 나타남.
특히 recall은 비교적으로 높으나, precision이 0.67에 머물렀음. 이는 모델이 phishing URL을 놓치는 문제보다는, 정상 URL을 phishing으로 잘못 분류하는 오탐이 많다는 것을 의미함.
이 결과는 phishing URL이 malware나 defacement URL처럼 명확한 악성 문자열 패턴을 갖기보다는, 정상 서비스에서도 자주 등장하는 토큰을 공유하는 경향이 있기 때문으로 해석할 수 있음.
즉, phishing URL은 정상 URL과 토큰 분포가 상대적으로 유사하여 단순한 위험 토큰 사전 기반 feature만으로는 구분이 어려움. 
추후 phishing 클래스에 대해 더 보수적인 decision threshold를 적용하거나, benign-phishing 간 확률 margin을 고려하는 방식, 또는 도메인 유사도나 브랜드 사칭 여부, URL 구조적 위장 패턴과 같은 phishing 특화 feature를 추가하는 방향으로 개선할 필요가 있음.

In [1]:
# ============================================================
# AI-based Multi-Class Malicious URL Detection
# Adaptive Risk Token Dictionary + Streaming Update
# Safe Full Version
# ============================================================

import os
import re
import json
import math
import urllib.parse
from collections import Counter, deque

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight


# ============================================================
# 0. Config
# ============================================================

SEED = 42
np.random.seed(SEED)

DATA_PATH = "./malicious_phish.csv"

URL_COL = "url"
LABEL_COL = "type"
BENIGN_LABEL = "benign"

# Risk dictionary
TOP_K_COMMON_MALICIOUS = 1000
TOP_K_PER_MAL_CLASS = 800
MIN_TOKEN_DF = 5
MIN_MAL_CLASSES_FOR_COMMON = 2
ALPHA = 1.0

# TF-IDF
MAX_TFIDF_FEATURES = 50000
TFIDF_MIN_DF = 3
TFIDF_MAX_DF = 0.95

# Streaming
BATCH_SIZE = 5000
UPDATE_INTERVAL = 3
SLIDING_WINDOW_BATCHES = 6

# Output
RISK_DICT_DIR = "./risk_dict"
RESULT_DIR = "./results"

os.makedirs(RISK_DICT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 1. Safe URL Tokenizer
# ============================================================

def clean_url_text(url):
    """
    어떤 URL 문자열이 들어와도 urlparse가 죽지 않도록 정리.
    특히 Invalid IPv6 URL을 유발하는 '[' ']' 제거.
    """
    if url is None:
        url = ""

    url = str(url).strip().lower()

    try:
        url = urllib.parse.unquote(url, errors="ignore")
    except Exception:
        pass

    # Invalid IPv6 URL 에러 방지 핵심
    url = url.replace("[", "")
    url = url.replace("]", "")

    # 제어 문자 제거
    url = re.sub(r"[\x00-\x1f\x7f]", "", url)

    return url


def split_by_separators(text):
    """
    URL 내부 문자열을 특수문자 기준으로 분리.
    """
    if text is None:
        return []

    text = clean_url_text(text)

    parts = re.split(r"[^a-zA-Z0-9]+", text)
    return [p for p in parts if len(p) >= 2]


def char_ngrams(token, n_min=3, n_max=4):
    """
    긴 문자열에 대해 character n-gram 생성.
    """
    token = str(token).lower()
    grams = []

    if len(token) < n_min:
        return grams

    for n in range(n_min, n_max + 1):
        if len(token) >= n:
            for i in range(len(token) - n + 1):
                grams.append(f"char:{token[i:i+n]}")

    return grams


def safe_parse_url(url):
    """
    urlparse를 안전하게 수행.
    실패하면 invalid.local로 대체.
    이 함수는 절대 에러를 밖으로 던지지 않음.
    """
    cleaned = clean_url_text(url)

    try:
        parse_target = cleaned

        if not re.match(r"^[a-zA-Z][a-zA-Z0-9+\-.]*://", parse_target):
            parse_target = "http://" + parse_target

        return urllib.parse.urlparse(parse_target)

    except Exception:
        try:
            return urllib.parse.urlparse("http://invalid.local/")
        except Exception:
            # 이론상 거의 도달하지 않지만 완전 방어용
            return None


def tokenize_url(url):
    """
    URL 구조 기반 보안 특화 tokenizer.
    어떤 URL이 들어와도 절대 예외를 밖으로 던지지 않음.
    """
    try:
        decoded_url = clean_url_text(url)
        parsed = safe_parse_url(decoded_url)

        tokens = []

        if parsed is None:
            general_parts = split_by_separators(decoded_url)
            for p in general_parts:
                tokens.append(f"tok:{p}")
            return tokens if tokens else ["empty_or_invalid_url"]

        host = parsed.netloc.lower() if parsed.netloc else ""
        path = parsed.path.lower() if parsed.path else ""
        query = parsed.query.lower() if parsed.query else ""

        # user:pass@host
        if "@" in host:
            tokens.append("has_userinfo")
            host = host.split("@")[-1]

        # port 제거
        if ":" in host:
            host = host.split(":")[0]

        # www 제거
        if host.startswith("www."):
            host = host[4:]

        # IP host 여부
        if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host):
            tokens.append("host_is_ip")

        # domain / subdomain / tld
        domain_parts = [p for p in host.split(".") if p]

        if domain_parts:
            tld = domain_parts[-1]
            tokens.append(f"tld:{tld}")

            for idx, part in enumerate(domain_parts):
                if idx == len(domain_parts) - 1:
                    tokens.append(f"tldpart:{part}")
                elif idx == len(domain_parts) - 2:
                    tokens.append(f"sld:{part}")
                else:
                    tokens.append(f"subdomain:{part}")

                tokens.append(f"domain:{part}")

                for sub in split_by_separators(part):
                    tokens.append(f"dpart:{sub}")

        # path
        path_segments = [seg for seg in path.split("/") if seg]

        for seg in path_segments:
            tokens.append(f"path:{seg}")

            for p in split_by_separators(seg):
                tokens.append(f"pseg:{p}")

        # extension
        if path_segments:
            last_seg = path_segments[-1]
            if "." in last_seg:
                ext = last_seg.split(".")[-1]
                if 1 <= len(ext) <= 8:
                    tokens.append(f"ext:{ext}")

        # query
        if query:
            tokens.append("has_query")

        try:
            query_pairs = urllib.parse.parse_qsl(query, keep_blank_values=True)
        except Exception:
            query_pairs = []

        for k, v in query_pairs:
            if k:
                tokens.append(f"qkey:{k}")
                for p in split_by_separators(k):
                    tokens.append(f"qk:{p}")

            if v:
                for p in split_by_separators(v):
                    tokens.append(f"qv:{p}")

        # 전체 URL 기준 일반 토큰
        general_parts = split_by_separators(decoded_url)

        for p in general_parts:
            tokens.append(f"tok:{p}")

        # 긴 문자열에 character n-gram 추가
        for p in general_parts:
            if len(p) >= 6:
                tokens.extend(char_ngrams(p, 3, 4))

        # 구조적 패턴 토큰
        if "-" in decoded_url:
            tokens.append("has_hyphen")
        if "_" in decoded_url:
            tokens.append("has_underscore")
        if "%" in decoded_url:
            tokens.append("has_percent_encoding")
        if decoded_url.count(".") >= 4:
            tokens.append("many_dots")
        if len(decoded_url) >= 100:
            tokens.append("very_long_url")

        if not tokens:
            tokens.append("empty_or_invalid_url")

        return tokens

    except Exception:
        return ["tokenizer_error"]


def safe_tokenize(url):
    """
    외부에서 항상 이 tokenizer를 사용.
    """
    try:
        return tokenize_url(url)
    except Exception:
        return ["tokenizer_error"]


# tokenizer 작동 테스트
test_urls = [
    "https://secure-login.example.com/account/verify?id=123",
    "http://abc.ru/download/app.apk",
    "paypal.com.verify-user-login-security.com/index.php",
    "http://[broken-ipv6-url",
    "[invalid]url.com/test"
]

print("Tokenizer test:")
for u in test_urls:
    print(u, "=>", safe_tokenize(u)[:20])


# ============================================================
# 2. Load Dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

print("\nOriginal shape:", df.shape)
print("Columns:", df.columns.tolist())

df = df[[URL_COL, LABEL_COL]].dropna()
df[URL_COL] = df[URL_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(str)

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())


# ============================================================
# 3. Split: Initial Train 50% / Stream Update 30% / Final Test 20%
# ============================================================

initial_train_df, temp_df = train_test_split(
    df,
    test_size=0.50,
    stratify=df[LABEL_COL],
    random_state=SEED
)

stream_update_df, final_test_df = train_test_split(
    temp_df,
    test_size=0.40,
    stratify=temp_df[LABEL_COL],
    random_state=SEED
)

print("\nSplit result")
print("Initial Train:", initial_train_df.shape)
print("Stream Update:", stream_update_df.shape)
print("Final Test:", final_test_df.shape)

print("\nInitial Train label distribution")
print(initial_train_df[LABEL_COL].value_counts())

print("\nStream Update label distribution")
print(stream_update_df[LABEL_COL].value_counts())

print("\nFinal Test label distribution")
print(final_test_df[LABEL_COL].value_counts())


# ============================================================
# 4. Label Encoding
# ============================================================

label_encoder = LabelEncoder()

y_initial = label_encoder.fit_transform(initial_train_df[LABEL_COL])
y_stream = label_encoder.transform(stream_update_df[LABEL_COL])
y_final = label_encoder.transform(final_test_df[LABEL_COL])

class_names = list(label_encoder.classes_)
malicious_classes = [c for c in class_names if c != BENIGN_LABEL]
all_class_ids = np.arange(len(class_names))

print("\nClass names:", class_names)
print("Malicious classes:", malicious_classes)


# ============================================================
# 5. Adaptive Risk Token Dictionary
# ============================================================

def build_adaptive_risk_dictionary(
    urls,
    labels,
    tokenizer,
    class_names,
    benign_label="benign",
    top_k_common=1000,
    top_k_per_class=800,
    min_df=5,
    min_mal_classes_for_common=2,
    alpha=1.0
):
    """
    위험 토큰 사전 생성.

    common_malicious:
        benign에서는 적게 나오고,
        여러 악성 클래스에서 공통적으로 많이 나오는 토큰.

    class_specific:
        특정 악성 클래스에서 다른 클래스보다 더 강하게 나타나는 토큰.
    """

    malicious_classes = [c for c in class_names if c != benign_label]

    class_token_df = {c: Counter() for c in class_names}
    total_token_df = Counter()
    class_doc_count = Counter()

    for url, label in zip(urls, labels):
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        class_doc_count[label] += 1

        for t in tokens:
            class_token_df[label][t] += 1
            total_token_df[t] += 1

    total_docs = sum(class_doc_count.values())
    benign_docs = class_doc_count[benign_label]
    malicious_docs = total_docs - benign_docs

    # --------------------------------------------------------
    # 5-1. Common malicious tokens
    # --------------------------------------------------------
    common_scores = {}

    for token, total_df in total_token_df.items():
        if total_df < min_df:
            continue

        benign_df = class_token_df[benign_label][token]
        malicious_df = sum(class_token_df[c][token] for c in malicious_classes)

        mal_class_presence = sum(
            1 for c in malicious_classes
            if class_token_df[c][token] > 0
        )

        if mal_class_presence < min_mal_classes_for_common:
            continue

        p_token_mal = (malicious_df + alpha) / (malicious_docs + 2 * alpha)
        p_token_benign = (benign_df + alpha) / (benign_docs + 2 * alpha)

        score = math.log(p_token_mal / p_token_benign)

        if score > 0:
            common_scores[token] = score

    common_malicious = dict(
        sorted(common_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_common]
    )

    # --------------------------------------------------------
    # 5-2. Class-specific malicious tokens
    # --------------------------------------------------------
    class_specific = {c: {} for c in malicious_classes}

    for c in malicious_classes:
        scores = {}

        n_c = class_doc_count[c]
        n_not_c = total_docs - n_c

        for token, total_df in total_token_df.items():
            if total_df < min_df:
                continue

            df_c = class_token_df[c][token]
            df_not_c = total_df - df_c

            p_token_c = (df_c + alpha) / (n_c + 2 * alpha)
            p_token_not_c = (df_not_c + alpha) / (n_not_c + 2 * alpha)

            score = math.log(p_token_c / p_token_not_c)

            if score > 0:
                scores[token] = score

        class_specific[c] = dict(
            sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_per_class]
        )

    return {
        "common_malicious": common_malicious,
        "class_specific": class_specific
    }


def save_risk_dict(risk_dict, version):
    path = os.path.join(RISK_DICT_DIR, f"risk_token_dict_v{version}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(risk_dict, f, ensure_ascii=False, indent=2)
    print(f"Saved risk dictionary: {path}")


def print_risk_dict_preview(risk_dict, top_n=15):
    print("\n===== Common Malicious Risk Tokens =====")
    for token, score in list(risk_dict["common_malicious"].items())[:top_n]:
        print(token, round(score, 4))

    print("\n===== Class-specific Risk Tokens =====")
    for c, d in risk_dict["class_specific"].items():
        print(f"\n[{c}]")
        for token, score in list(d.items())[:top_n]:
            print(token, round(score, 4))


# ============================================================
# 6. Build Initial Risk Dictionary
# ============================================================

print("\nBuilding initial risk dictionary...")

initial_urls = initial_train_df[URL_COL].tolist()
initial_labels = initial_train_df[LABEL_COL].tolist()

risk_dict = build_adaptive_risk_dictionary(
    urls=initial_urls,
    labels=initial_labels,
    tokenizer=safe_tokenize,
    class_names=class_names,
    benign_label=BENIGN_LABEL,
    top_k_common=TOP_K_COMMON_MALICIOUS,
    top_k_per_class=TOP_K_PER_MAL_CLASS,
    min_df=MIN_TOKEN_DF,
    min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
    alpha=ALPHA
)

save_risk_dict(risk_dict, version=0)
print_risk_dict_preview(risk_dict, top_n=20)


# ============================================================
# 7. Feature Engineering
# ============================================================

def risk_score_features(urls, risk_dict, tokenizer, malicious_classes):
    """
    URL 하나를 위험 토큰 사전 기반 feature로 변환.

    feature 구성:
    - common malicious score
    - common malicious match count
    - 각 악성 클래스별 score
    - 각 악성 클래스별 match count
    """
    rows = []

    common_dict = risk_dict["common_malicious"]
    class_dict = risk_dict["class_specific"]

    for url in urls:
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        row = []

        common_score = 0.0
        common_count = 0

        for t in tokens:
            if t in common_dict:
                common_score += common_dict[t]
                common_count += 1

        row.append(common_score)
        row.append(common_count)

        for c in malicious_classes:
            c_score = 0.0
            c_count = 0

            c_dict = class_dict.get(c, {})

            for t in tokens:
                if t in c_dict:
                    c_score += c_dict[t]
                    c_count += 1

            row.append(c_score)
            row.append(c_count)

        rows.append(row)

    return np.array(rows, dtype=np.float32)


def lexical_features(urls):
    """
    URL 구조 통계 feature.
    """
    rows = []

    for url in urls:
        u = clean_url_text(url)
        length = len(u)

        digit_count = sum(ch.isdigit() for ch in u)
        alpha_count = sum(ch.isalpha() for ch in u)
        special_count = sum(not ch.isalnum() for ch in u)

        dot_count = u.count(".")
        slash_count = u.count("/")
        hyphen_count = u.count("-")
        underscore_count = u.count("_")
        question_count = u.count("?")
        equal_count = u.count("=")
        amp_count = u.count("&")
        percent_count = u.count("%")
        at_count = u.count("@")

        digit_ratio = digit_count / max(length, 1)
        alpha_ratio = alpha_count / max(length, 1)
        special_ratio = special_count / max(length, 1)

        has_ip = 1 if re.search(r"\d{1,3}(\.\d{1,3}){3}", u) else 0
        has_https = 1 if u.startswith("https://") else 0
        has_http = 1 if u.startswith("http://") else 0

        rows.append([
            length,
            digit_count,
            alpha_count,
            special_count,
            dot_count,
            slash_count,
            hyphen_count,
            underscore_count,
            question_count,
            equal_count,
            amp_count,
            percent_count,
            at_count,
            digit_ratio,
            alpha_ratio,
            special_ratio,
            has_ip,
            has_https,
            has_http
        ])

    return np.array(rows, dtype=np.float32)


def build_train_features(urls, risk_dict, malicious_classes):
    """
    Initial train용 feature 생성.
    TF-IDF vectorizer와 scaler는 initial train에 대해서만 fit.
    """
    vectorizer = TfidfVectorizer(
        tokenizer=safe_tokenize,
        token_pattern=None,
        lowercase=False,
        max_features=MAX_TFIDF_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF
    )

    X_tfidf = vectorizer.fit_transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])

    scaler = StandardScaler()
    X_extra_scaled = scaler.fit_transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X, vectorizer, scaler


def build_eval_features(urls, risk_dict, malicious_classes, vectorizer, scaler):
    """
    Stream/Test용 feature 생성.
    vectorizer와 scaler는 initial train에서 fit한 것을 그대로 사용.
    """
    X_tfidf = vectorizer.transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])
    X_extra_scaled = scaler.transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X


# ============================================================
# 8. Initial Model Training
# ============================================================

print("\nBuilding initial features...")

X_initial, vectorizer, scaler = build_train_features(
    urls=initial_train_df[URL_COL].tolist(),
    risk_dict=risk_dict,
    malicious_classes=malicious_classes
)

print("X_initial shape:", X_initial.shape)

model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    n_jobs=-1
)

initial_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_initial
)

model.partial_fit(
    X_initial,
    y_initial,
    classes=all_class_ids,
    sample_weight=initial_sample_weight
)

print("\nInitial model trained.")


# ============================================================
# 9. Evaluation Function
# ============================================================

def evaluate_model(
    model,
    urls,
    y_true,
    risk_dict,
    malicious_classes,
    vectorizer,
    scaler,
    title="Evaluation",
    verbose=True
):
    X = build_eval_features(
        urls=urls,
        risk_dict=risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=vectorizer,
        scaler=scaler
    )

    y_pred = model.predict(X)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    if verbose:
        print(f"\n===== {title} =====")
        print("Accuracy:", round(acc, 4))
        print("Macro-F1:", round(macro_f1, 4))
        print("Weighted-F1:", round(weighted_f1, 4))
        print("\nClassification Report")
        print(classification_report(y_true, y_pred, target_names=class_names))
        print("\nConfusion Matrix")
        print(confusion_matrix(y_true, y_pred))

    return {
        "title": title,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }


# ============================================================
# 10. Stream Utility Functions
# ============================================================

def make_batches(df, batch_size):
    batches = []

    n = len(df)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batches.append(df.iloc[start:end].copy())

    return batches


def summarize_risk_dict_change(old_dict, new_dict, malicious_classes):
    summary = {}

    old_common = set(old_dict["common_malicious"].keys())
    new_common = set(new_dict["common_malicious"].keys())

    summary["common_malicious"] = {
        "added": len(new_common - old_common),
        "removed": len(old_common - new_common),
        "kept": len(old_common & new_common),
        "sample_added": list(new_common - old_common)[:10],
        "sample_removed": list(old_common - new_common)[:10]
    }

    for c in malicious_classes:
        old_tokens = set(old_dict["class_specific"].get(c, {}).keys())
        new_tokens = set(new_dict["class_specific"].get(c, {}).keys())

        summary[c] = {
            "added": len(new_tokens - old_tokens),
            "removed": len(old_tokens - new_tokens),
            "kept": len(old_tokens & new_tokens),
            "sample_added": list(new_tokens - old_tokens)[:10],
            "sample_removed": list(old_tokens - new_tokens)[:10]
        }

    return summary


def print_change_summary(summary):
    print("\nRisk Dictionary Change Summary")

    for key, value in summary.items():
        print(f"\n[{key}]")
        print("added:", value["added"])
        print("removed:", value["removed"])
        print("kept:", value["kept"])
        print("sample_added:", value["sample_added"])
        print("sample_removed:", value["sample_removed"])


# ============================================================
# 11. Streaming Update
# ============================================================

stream_batches = make_batches(stream_update_df, BATCH_SIZE)

print("\nNumber of stream batches:", len(stream_batches))
print("Batch size:", BATCH_SIZE)
print("Update interval:", UPDATE_INTERVAL)
print("Sliding window batches:", SLIDING_WINDOW_BATCHES)

current_model = model
current_risk_dict = risk_dict
current_vectorizer = vectorizer
current_scaler = scaler

recent_stream_buffer = deque(maxlen=SLIDING_WINDOW_BATCHES)

stream_results = []
version = 0

for batch_idx, batch_df in enumerate(stream_batches, start=1):
    print(f"\n\n========== Stream Batch {batch_idx}/{len(stream_batches)} ==========")

    recent_stream_buffer.append(batch_df)

    batch_urls = batch_df[URL_COL].tolist()
    batch_y = label_encoder.transform(batch_df[LABEL_COL])

    # 현재 모델로 stream batch 평가
    # Final Test는 여기서 사용하지 않음.
    batch_result = evaluate_model(
        model=current_model,
        urls=batch_urls,
        y_true=batch_y,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler,
        title=f"Before Update - Stream Batch {batch_idx}",
        verbose=False
    )

    print(
        f"Before Update Batch {batch_idx} | "
        f"ACC={batch_result['accuracy']:.4f}, "
        f"Macro-F1={batch_result['macro_f1']:.4f}, "
        f"Weighted-F1={batch_result['weighted_f1']:.4f}"
    )

    batch_result["batch_idx"] = batch_idx
    batch_result["dict_version"] = version
    stream_results.append(batch_result)

    # 현재 batch로 classifier incremental update
    X_batch = build_eval_features(
        urls=batch_urls,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler
    )

    batch_sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=batch_y
    )

    current_model.partial_fit(
        X_batch,
        batch_y,
        classes=all_class_ids,
        sample_weight=batch_sample_weight
    )

    # 일정 주기마다 위험 토큰 사전 갱신
    if batch_idx % UPDATE_INTERVAL == 0:
        print(f"\n----- Risk Dictionary Update at Batch {batch_idx} -----")

        version += 1

        recent_df = pd.concat(list(recent_stream_buffer), axis=0)

        # Initial Train + 최근 Stream Window로만 사전 갱신
        # Final Test는 절대 포함하지 않음.
        dict_update_df = pd.concat([initial_train_df, recent_df], axis=0)

        update_urls = dict_update_df[URL_COL].tolist()
        update_labels = dict_update_df[LABEL_COL].tolist()

        new_risk_dict = build_adaptive_risk_dictionary(
            urls=update_urls,
            labels=update_labels,
            tokenizer=safe_tokenize,
            class_names=class_names,
            benign_label=BENIGN_LABEL,
            top_k_common=TOP_K_COMMON_MALICIOUS,
            top_k_per_class=TOP_K_PER_MAL_CLASS,
            min_df=MIN_TOKEN_DF,
            min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
            alpha=ALPHA
        )

        change_summary = summarize_risk_dict_change(
            old_dict=current_risk_dict,
            new_dict=new_risk_dict,
            malicious_classes=malicious_classes
        )

        print_change_summary(change_summary)
        save_risk_dict(new_risk_dict, version=version)

        current_risk_dict = new_risk_dict

        # 새 risk dictionary 기준으로 최근 stream window를 한 번 더 학습
        X_recent = build_eval_features(
            urls=recent_df[URL_COL].tolist(),
            risk_dict=current_risk_dict,
            malicious_classes=malicious_classes,
            vectorizer=current_vectorizer,
            scaler=current_scaler
        )

        y_recent = label_encoder.transform(recent_df[LABEL_COL])

        recent_sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_recent
        )

        current_model.partial_fit(
            X_recent,
            y_recent,
            classes=all_class_ids,
            sample_weight=recent_sample_weight
        )

        print(f"Dictionary version updated to v{version}")


# ============================================================
# 12. Final Test Evaluation
# ============================================================

print("\n\n================================================")
print("Final Test Evaluation")
print("================================================")

final_urls = final_test_df[URL_COL].tolist()

final_result = evaluate_model(
    model=current_model,
    urls=final_urls,
    y_true=y_final,
    risk_dict=current_risk_dict,
    malicious_classes=malicious_classes,
    vectorizer=current_vectorizer,
    scaler=current_scaler,
    title="Final Adaptive Model on Final Test",
    verbose=True
)

# 결과 저장
stream_results_df = pd.DataFrame(stream_results)
stream_result_path = os.path.join(RESULT_DIR, "stream_update_results.csv")
stream_results_df.to_csv(stream_result_path, index=False)

final_result_path = os.path.join(RESULT_DIR, "final_test_result.json")
with open(final_result_path, "w", encoding="utf-8") as f:
    json.dump(final_result, f, ensure_ascii=False, indent=2)

print("\nSaved stream results:", stream_result_path)
print("Saved final test result:", final_result_path)

print("\nDone.")

Tokenizer test:
https://secure-login.example.com/account/verify?id=123 => ['tld:com', 'subdomain:secure-login', 'domain:secure-login', 'dpart:secure', 'dpart:login', 'sld:example', 'domain:example', 'dpart:example', 'tldpart:com', 'domain:com', 'dpart:com', 'path:account', 'pseg:account', 'path:verify', 'pseg:verify', 'has_query', 'qkey:id', 'qk:id', 'qv:123', 'tok:https']
http://abc.ru/download/app.apk => ['tld:ru', 'sld:abc', 'domain:abc', 'dpart:abc', 'tldpart:ru', 'domain:ru', 'dpart:ru', 'path:download', 'pseg:download', 'path:app.apk', 'pseg:app', 'pseg:apk', 'ext:apk', 'tok:http', 'tok:abc', 'tok:ru', 'tok:download', 'tok:app', 'tok:apk', 'char:dow']
paypal.com.verify-user-login-security.com/index.php => ['tld:com', 'subdomain:paypal', 'domain:paypal', 'dpart:paypal', 'subdomain:com', 'domain:com', 'dpart:com', 'sld:verify-user-login-security', 'domain:verify-user-login-security', 'dpart:verify', 'dpart:user', 'dpart:login', 'dpart:security', 'tldpart:com', 'domain:com', 'dpart:

data split 비율 수정 

In [2]:
# ============================================================
# AI-based Multi-Class Malicious URL Detection
# Adaptive Risk Token Dictionary + Streaming Update
# Safe Full Version
# ============================================================

import os
import re
import json
import math
import urllib.parse
from collections import Counter, deque

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight


# ============================================================
# 0. Config
# ============================================================

SEED = 42
np.random.seed(SEED)

DATA_PATH = "./malicious_phish.csv"

URL_COL = "url"
LABEL_COL = "type"
BENIGN_LABEL = "benign"

# Risk dictionary
TOP_K_COMMON_MALICIOUS = 1000
TOP_K_PER_MAL_CLASS = 800
MIN_TOKEN_DF = 5
MIN_MAL_CLASSES_FOR_COMMON = 2
ALPHA = 1.0

# TF-IDF
MAX_TFIDF_FEATURES = 50000
TFIDF_MIN_DF = 3
TFIDF_MAX_DF = 0.95

# Streaming
BATCH_SIZE = 5000
UPDATE_INTERVAL = 3
SLIDING_WINDOW_BATCHES = 6

# Output
RISK_DICT_DIR = "./risk_dict"
RESULT_DIR = "./results"

os.makedirs(RISK_DICT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 1. Safe URL Tokenizer
# ============================================================

def clean_url_text(url):
    """
    어떤 URL 문자열이 들어와도 urlparse가 죽지 않도록 정리.
    특히 Invalid IPv6 URL을 유발하는 '[' ']' 제거.
    """
    if url is None:
        url = ""

    url = str(url).strip().lower()

    try:
        url = urllib.parse.unquote(url, errors="ignore")
    except Exception:
        pass

    # Invalid IPv6 URL 에러 방지 핵심
    url = url.replace("[", "")
    url = url.replace("]", "")

    # 제어 문자 제거
    url = re.sub(r"[\x00-\x1f\x7f]", "", url)

    return url


def split_by_separators(text):
    """
    URL 내부 문자열을 특수문자 기준으로 분리.
    """
    if text is None:
        return []

    text = clean_url_text(text)

    parts = re.split(r"[^a-zA-Z0-9]+", text)
    return [p for p in parts if len(p) >= 2]


def char_ngrams(token, n_min=3, n_max=4):
    """
    긴 문자열에 대해 character n-gram 생성.
    """
    token = str(token).lower()
    grams = []

    if len(token) < n_min:
        return grams

    for n in range(n_min, n_max + 1):
        if len(token) >= n:
            for i in range(len(token) - n + 1):
                grams.append(f"char:{token[i:i+n]}")

    return grams


def safe_parse_url(url):
    """
    urlparse를 안전하게 수행.
    실패하면 invalid.local로 대체.
    이 함수는 절대 에러를 밖으로 던지지 않음.
    """
    cleaned = clean_url_text(url)

    try:
        parse_target = cleaned

        if not re.match(r"^[a-zA-Z][a-zA-Z0-9+\-.]*://", parse_target):
            parse_target = "http://" + parse_target

        return urllib.parse.urlparse(parse_target)

    except Exception:
        try:
            return urllib.parse.urlparse("http://invalid.local/")
        except Exception:
            # 이론상 거의 도달하지 않지만 완전 방어용
            return None


def tokenize_url(url):
    """
    URL 구조 기반 보안 특화 tokenizer.
    어떤 URL이 들어와도 절대 예외를 밖으로 던지지 않음.
    """
    try:
        decoded_url = clean_url_text(url)
        parsed = safe_parse_url(decoded_url)

        tokens = []

        if parsed is None:
            general_parts = split_by_separators(decoded_url)
            for p in general_parts:
                tokens.append(f"tok:{p}")
            return tokens if tokens else ["empty_or_invalid_url"]

        host = parsed.netloc.lower() if parsed.netloc else ""
        path = parsed.path.lower() if parsed.path else ""
        query = parsed.query.lower() if parsed.query else ""

        # user:pass@host
        if "@" in host:
            tokens.append("has_userinfo")
            host = host.split("@")[-1]

        # port 제거
        if ":" in host:
            host = host.split(":")[0]

        # www 제거
        if host.startswith("www."):
            host = host[4:]

        # IP host 여부
        if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host):
            tokens.append("host_is_ip")

        # domain / subdomain / tld
        domain_parts = [p for p in host.split(".") if p]

        if domain_parts:
            tld = domain_parts[-1]
            tokens.append(f"tld:{tld}")

            for idx, part in enumerate(domain_parts):
                if idx == len(domain_parts) - 1:
                    tokens.append(f"tldpart:{part}")
                elif idx == len(domain_parts) - 2:
                    tokens.append(f"sld:{part}")
                else:
                    tokens.append(f"subdomain:{part}")

                tokens.append(f"domain:{part}")

                for sub in split_by_separators(part):
                    tokens.append(f"dpart:{sub}")

        # path
        path_segments = [seg for seg in path.split("/") if seg]

        for seg in path_segments:
            tokens.append(f"path:{seg}")

            for p in split_by_separators(seg):
                tokens.append(f"pseg:{p}")

        # extension
        if path_segments:
            last_seg = path_segments[-1]
            if "." in last_seg:
                ext = last_seg.split(".")[-1]
                if 1 <= len(ext) <= 8:
                    tokens.append(f"ext:{ext}")

        # query
        if query:
            tokens.append("has_query")

        try:
            query_pairs = urllib.parse.parse_qsl(query, keep_blank_values=True)
        except Exception:
            query_pairs = []

        for k, v in query_pairs:
            if k:
                tokens.append(f"qkey:{k}")
                for p in split_by_separators(k):
                    tokens.append(f"qk:{p}")

            if v:
                for p in split_by_separators(v):
                    tokens.append(f"qv:{p}")

        # 전체 URL 기준 일반 토큰
        general_parts = split_by_separators(decoded_url)

        for p in general_parts:
            tokens.append(f"tok:{p}")

        # 긴 문자열에 character n-gram 추가
        for p in general_parts:
            if len(p) >= 6:
                tokens.extend(char_ngrams(p, 3, 4))

        # 구조적 패턴 토큰
        if "-" in decoded_url:
            tokens.append("has_hyphen")
        if "_" in decoded_url:
            tokens.append("has_underscore")
        if "%" in decoded_url:
            tokens.append("has_percent_encoding")
        if decoded_url.count(".") >= 4:
            tokens.append("many_dots")
        if len(decoded_url) >= 100:
            tokens.append("very_long_url")

        if not tokens:
            tokens.append("empty_or_invalid_url")

        return tokens

    except Exception:
        return ["tokenizer_error"]


def safe_tokenize(url):
    """
    외부에서 항상 이 tokenizer를 사용.
    """
    try:
        return tokenize_url(url)
    except Exception:
        return ["tokenizer_error"]


# tokenizer 작동 테스트
test_urls = [
    "https://secure-login.example.com/account/verify?id=123",
    "http://abc.ru/download/app.apk",
    "paypal.com.verify-user-login-security.com/index.php",
    "http://[broken-ipv6-url",
    "[invalid]url.com/test"
]

print("Tokenizer test:")
for u in test_urls:
    print(u, "=>", safe_tokenize(u)[:20])


# ============================================================
# 2. Load Dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

print("\nOriginal shape:", df.shape)
print("Columns:", df.columns.tolist())

df = df[[URL_COL, LABEL_COL]].dropna()
df[URL_COL] = df[URL_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(str)

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())


# ============================================================
# 3. Split: Initial Train 40% / Stream Update 30% / Final Test 30%
# ============================================================

# First split:
# 40% -> Initial Train
# 60% -> Temporary pool for Stream Update + Final Test
initial_train_df, temp_df = train_test_split(
    df,
    test_size=0.60,
    stratify=df[LABEL_COL],
    random_state=SEED
)

# Second split:
# temp_df is 60% of total.
# Split it equally:
# 30% -> Stream Update
# 30% -> Final Test
stream_update_df, final_test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[LABEL_COL],
    random_state=SEED
)

print("\nSplit result")
print("Initial Train:", initial_train_df.shape)
print("Stream Update:", stream_update_df.shape)
print("Final Test:", final_test_df.shape)

print("\nInitial Train label distribution")
print(initial_train_df[LABEL_COL].value_counts())

print("\nStream Update label distribution")
print(stream_update_df[LABEL_COL].value_counts())

print("\nFinal Test label distribution")
print(final_test_df[LABEL_COL].value_counts())


# ============================================================
# 4. Label Encoding
# ============================================================

label_encoder = LabelEncoder()

y_initial = label_encoder.fit_transform(initial_train_df[LABEL_COL])
y_stream = label_encoder.transform(stream_update_df[LABEL_COL])
y_final = label_encoder.transform(final_test_df[LABEL_COL])

class_names = list(label_encoder.classes_)
malicious_classes = [c for c in class_names if c != BENIGN_LABEL]
all_class_ids = np.arange(len(class_names))

print("\nClass names:", class_names)
print("Malicious classes:", malicious_classes)


# ============================================================
# 5. Adaptive Risk Token Dictionary
# ============================================================

def build_adaptive_risk_dictionary(
    urls,
    labels,
    tokenizer,
    class_names,
    benign_label="benign",
    top_k_common=1000,
    top_k_per_class=800,
    min_df=5,
    min_mal_classes_for_common=2,
    alpha=1.0
):
    """
    위험 토큰 사전 생성.

    common_malicious:
        benign에서는 적게 나오고,
        여러 악성 클래스에서 공통적으로 많이 나오는 토큰.

    class_specific:
        특정 악성 클래스에서 다른 클래스보다 더 강하게 나타나는 토큰.
    """

    malicious_classes = [c for c in class_names if c != benign_label]

    class_token_df = {c: Counter() for c in class_names}
    total_token_df = Counter()
    class_doc_count = Counter()

    for url, label in zip(urls, labels):
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        class_doc_count[label] += 1

        for t in tokens:
            class_token_df[label][t] += 1
            total_token_df[t] += 1

    total_docs = sum(class_doc_count.values())
    benign_docs = class_doc_count[benign_label]
    malicious_docs = total_docs - benign_docs

    # --------------------------------------------------------
    # 5-1. Common malicious tokens
    # --------------------------------------------------------
    common_scores = {}

    for token, total_df in total_token_df.items():
        if total_df < min_df:
            continue

        benign_df = class_token_df[benign_label][token]
        malicious_df = sum(class_token_df[c][token] for c in malicious_classes)

        mal_class_presence = sum(
            1 for c in malicious_classes
            if class_token_df[c][token] > 0
        )

        if mal_class_presence < min_mal_classes_for_common:
            continue

        p_token_mal = (malicious_df + alpha) / (malicious_docs + 2 * alpha)
        p_token_benign = (benign_df + alpha) / (benign_docs + 2 * alpha)

        score = math.log(p_token_mal / p_token_benign)

        if score > 0:
            common_scores[token] = score

    common_malicious = dict(
        sorted(common_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_common]
    )

    # --------------------------------------------------------
    # 5-2. Class-specific malicious tokens
    # --------------------------------------------------------
    class_specific = {c: {} for c in malicious_classes}

    for c in malicious_classes:
        scores = {}

        n_c = class_doc_count[c]
        n_not_c = total_docs - n_c

        for token, total_df in total_token_df.items():
            if total_df < min_df:
                continue

            df_c = class_token_df[c][token]
            df_not_c = total_df - df_c

            p_token_c = (df_c + alpha) / (n_c + 2 * alpha)
            p_token_not_c = (df_not_c + alpha) / (n_not_c + 2 * alpha)

            score = math.log(p_token_c / p_token_not_c)

            if score > 0:
                scores[token] = score

        class_specific[c] = dict(
            sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_per_class]
        )

    return {
        "common_malicious": common_malicious,
        "class_specific": class_specific
    }


def save_risk_dict(risk_dict, version):
    path = os.path.join(RISK_DICT_DIR, f"risk_token_dict_v{version}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(risk_dict, f, ensure_ascii=False, indent=2)
    print(f"Saved risk dictionary: {path}")


def print_risk_dict_preview(risk_dict, top_n=15):
    print("\n===== Common Malicious Risk Tokens =====")
    for token, score in list(risk_dict["common_malicious"].items())[:top_n]:
        print(token, round(score, 4))

    print("\n===== Class-specific Risk Tokens =====")
    for c, d in risk_dict["class_specific"].items():
        print(f"\n[{c}]")
        for token, score in list(d.items())[:top_n]:
            print(token, round(score, 4))


# ============================================================
# 6. Build Initial Risk Dictionary
# ============================================================

print("\nBuilding initial risk dictionary...")

initial_urls = initial_train_df[URL_COL].tolist()
initial_labels = initial_train_df[LABEL_COL].tolist()

risk_dict = build_adaptive_risk_dictionary(
    urls=initial_urls,
    labels=initial_labels,
    tokenizer=safe_tokenize,
    class_names=class_names,
    benign_label=BENIGN_LABEL,
    top_k_common=TOP_K_COMMON_MALICIOUS,
    top_k_per_class=TOP_K_PER_MAL_CLASS,
    min_df=MIN_TOKEN_DF,
    min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
    alpha=ALPHA
)

save_risk_dict(risk_dict, version=0)
print_risk_dict_preview(risk_dict, top_n=20)


# ============================================================
# 7. Feature Engineering
# ============================================================

def risk_score_features(urls, risk_dict, tokenizer, malicious_classes):
    """
    URL 하나를 위험 토큰 사전 기반 feature로 변환.

    feature 구성:
    - common malicious score
    - common malicious match count
    - 각 악성 클래스별 score
    - 각 악성 클래스별 match count
    """
    rows = []

    common_dict = risk_dict["common_malicious"]
    class_dict = risk_dict["class_specific"]

    for url in urls:
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        row = []

        common_score = 0.0
        common_count = 0

        for t in tokens:
            if t in common_dict:
                common_score += common_dict[t]
                common_count += 1

        row.append(common_score)
        row.append(common_count)

        for c in malicious_classes:
            c_score = 0.0
            c_count = 0

            c_dict = class_dict.get(c, {})

            for t in tokens:
                if t in c_dict:
                    c_score += c_dict[t]
                    c_count += 1

            row.append(c_score)
            row.append(c_count)

        rows.append(row)

    return np.array(rows, dtype=np.float32)


def lexical_features(urls):
    """
    URL 구조 통계 feature.
    """
    rows = []

    for url in urls:
        u = clean_url_text(url)
        length = len(u)

        digit_count = sum(ch.isdigit() for ch in u)
        alpha_count = sum(ch.isalpha() for ch in u)
        special_count = sum(not ch.isalnum() for ch in u)

        dot_count = u.count(".")
        slash_count = u.count("/")
        hyphen_count = u.count("-")
        underscore_count = u.count("_")
        question_count = u.count("?")
        equal_count = u.count("=")
        amp_count = u.count("&")
        percent_count = u.count("%")
        at_count = u.count("@")

        digit_ratio = digit_count / max(length, 1)
        alpha_ratio = alpha_count / max(length, 1)
        special_ratio = special_count / max(length, 1)

        has_ip = 1 if re.search(r"\d{1,3}(\.\d{1,3}){3}", u) else 0
        has_https = 1 if u.startswith("https://") else 0
        has_http = 1 if u.startswith("http://") else 0

        rows.append([
            length,
            digit_count,
            alpha_count,
            special_count,
            dot_count,
            slash_count,
            hyphen_count,
            underscore_count,
            question_count,
            equal_count,
            amp_count,
            percent_count,
            at_count,
            digit_ratio,
            alpha_ratio,
            special_ratio,
            has_ip,
            has_https,
            has_http
        ])

    return np.array(rows, dtype=np.float32)


def build_train_features(urls, risk_dict, malicious_classes):
    """
    Initial train용 feature 생성.
    TF-IDF vectorizer와 scaler는 initial train에 대해서만 fit.
    """
    vectorizer = TfidfVectorizer(
        tokenizer=safe_tokenize,
        token_pattern=None,
        lowercase=False,
        max_features=MAX_TFIDF_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF
    )

    X_tfidf = vectorizer.fit_transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])

    scaler = StandardScaler()
    X_extra_scaled = scaler.fit_transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X, vectorizer, scaler


def build_eval_features(urls, risk_dict, malicious_classes, vectorizer, scaler):
    """
    Stream/Test용 feature 생성.
    vectorizer와 scaler는 initial train에서 fit한 것을 그대로 사용.
    """
    X_tfidf = vectorizer.transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])
    X_extra_scaled = scaler.transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X


# ============================================================
# 8. Initial Model Training
# ============================================================

print("\nBuilding initial features...")

X_initial, vectorizer, scaler = build_train_features(
    urls=initial_train_df[URL_COL].tolist(),
    risk_dict=risk_dict,
    malicious_classes=malicious_classes
)

print("X_initial shape:", X_initial.shape)

model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    n_jobs=-1
)

initial_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_initial
)

model.partial_fit(
    X_initial,
    y_initial,
    classes=all_class_ids,
    sample_weight=initial_sample_weight
)

print("\nInitial model trained.")


# ============================================================
# 9. Evaluation Function
# ============================================================

def evaluate_model(
    model,
    urls,
    y_true,
    risk_dict,
    malicious_classes,
    vectorizer,
    scaler,
    title="Evaluation",
    verbose=True
):
    X = build_eval_features(
        urls=urls,
        risk_dict=risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=vectorizer,
        scaler=scaler
    )

    y_pred = model.predict(X)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    if verbose:
        print(f"\n===== {title} =====")
        print("Accuracy:", round(acc, 4))
        print("Macro-F1:", round(macro_f1, 4))
        print("Weighted-F1:", round(weighted_f1, 4))
        print("\nClassification Report")
        print(classification_report(y_true, y_pred, target_names=class_names))
        print("\nConfusion Matrix")
        print(confusion_matrix(y_true, y_pred))

    return {
        "title": title,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }


# ============================================================
# 10. Stream Utility Functions
# ============================================================

def make_batches(df, batch_size):
    batches = []

    n = len(df)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batches.append(df.iloc[start:end].copy())

    return batches


def summarize_risk_dict_change(old_dict, new_dict, malicious_classes):
    summary = {}

    old_common = set(old_dict["common_malicious"].keys())
    new_common = set(new_dict["common_malicious"].keys())

    summary["common_malicious"] = {
        "added": len(new_common - old_common),
        "removed": len(old_common - new_common),
        "kept": len(old_common & new_common),
        "sample_added": list(new_common - old_common)[:10],
        "sample_removed": list(old_common - new_common)[:10]
    }

    for c in malicious_classes:
        old_tokens = set(old_dict["class_specific"].get(c, {}).keys())
        new_tokens = set(new_dict["class_specific"].get(c, {}).keys())

        summary[c] = {
            "added": len(new_tokens - old_tokens),
            "removed": len(old_tokens - new_tokens),
            "kept": len(old_tokens & new_tokens),
            "sample_added": list(new_tokens - old_tokens)[:10],
            "sample_removed": list(old_tokens - new_tokens)[:10]
        }

    return summary


def print_change_summary(summary):
    print("\nRisk Dictionary Change Summary")

    for key, value in summary.items():
        print(f"\n[{key}]")
        print("added:", value["added"])
        print("removed:", value["removed"])
        print("kept:", value["kept"])
        print("sample_added:", value["sample_added"])
        print("sample_removed:", value["sample_removed"])


# ============================================================
# 11. Streaming Update
# ============================================================

stream_batches = make_batches(stream_update_df, BATCH_SIZE)

print("\nNumber of stream batches:", len(stream_batches))
print("Batch size:", BATCH_SIZE)
print("Update interval:", UPDATE_INTERVAL)
print("Sliding window batches:", SLIDING_WINDOW_BATCHES)

current_model = model
current_risk_dict = risk_dict
current_vectorizer = vectorizer
current_scaler = scaler

recent_stream_buffer = deque(maxlen=SLIDING_WINDOW_BATCHES)

stream_results = []
version = 0

for batch_idx, batch_df in enumerate(stream_batches, start=1):
    print(f"\n\n========== Stream Batch {batch_idx}/{len(stream_batches)} ==========")

    recent_stream_buffer.append(batch_df)

    batch_urls = batch_df[URL_COL].tolist()
    batch_y = label_encoder.transform(batch_df[LABEL_COL])

    # 현재 모델로 stream batch 평가
    # Final Test는 여기서 사용하지 않음.
    batch_result = evaluate_model(
        model=current_model,
        urls=batch_urls,
        y_true=batch_y,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler,
        title=f"Before Update - Stream Batch {batch_idx}",
        verbose=False
    )

    print(
        f"Before Update Batch {batch_idx} | "
        f"ACC={batch_result['accuracy']:.4f}, "
        f"Macro-F1={batch_result['macro_f1']:.4f}, "
        f"Weighted-F1={batch_result['weighted_f1']:.4f}"
    )

    batch_result["batch_idx"] = batch_idx
    batch_result["dict_version"] = version
    stream_results.append(batch_result)

    # 현재 batch로 classifier incremental update
    X_batch = build_eval_features(
        urls=batch_urls,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler
    )

    batch_sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=batch_y
    )

    current_model.partial_fit(
        X_batch,
        batch_y,
        classes=all_class_ids,
        sample_weight=batch_sample_weight
    )

    # 일정 주기마다 위험 토큰 사전 갱신
    if batch_idx % UPDATE_INTERVAL == 0:
        print(f"\n----- Risk Dictionary Update at Batch {batch_idx} -----")

        version += 1

        recent_df = pd.concat(list(recent_stream_buffer), axis=0)

        # Initial Train + 최근 Stream Window로만 사전 갱신
        # Final Test는 절대 포함하지 않음.
        dict_update_df = pd.concat([initial_train_df, recent_df], axis=0)

        update_urls = dict_update_df[URL_COL].tolist()
        update_labels = dict_update_df[LABEL_COL].tolist()

        new_risk_dict = build_adaptive_risk_dictionary(
            urls=update_urls,
            labels=update_labels,
            tokenizer=safe_tokenize,
            class_names=class_names,
            benign_label=BENIGN_LABEL,
            top_k_common=TOP_K_COMMON_MALICIOUS,
            top_k_per_class=TOP_K_PER_MAL_CLASS,
            min_df=MIN_TOKEN_DF,
            min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
            alpha=ALPHA
        )

        change_summary = summarize_risk_dict_change(
            old_dict=current_risk_dict,
            new_dict=new_risk_dict,
            malicious_classes=malicious_classes
        )

        print_change_summary(change_summary)
        save_risk_dict(new_risk_dict, version=version)

        current_risk_dict = new_risk_dict

        # 새 risk dictionary 기준으로 최근 stream window를 한 번 더 학습
        X_recent = build_eval_features(
            urls=recent_df[URL_COL].tolist(),
            risk_dict=current_risk_dict,
            malicious_classes=malicious_classes,
            vectorizer=current_vectorizer,
            scaler=current_scaler
        )

        y_recent = label_encoder.transform(recent_df[LABEL_COL])

        recent_sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_recent
        )

        current_model.partial_fit(
            X_recent,
            y_recent,
            classes=all_class_ids,
            sample_weight=recent_sample_weight
        )

        print(f"Dictionary version updated to v{version}")


# ============================================================
# 12. Final Test Evaluation
# ============================================================

print("\n\n================================================")
print("Final Test Evaluation")
print("================================================")

final_urls = final_test_df[URL_COL].tolist()

final_result = evaluate_model(
    model=current_model,
    urls=final_urls,
    y_true=y_final,
    risk_dict=current_risk_dict,
    malicious_classes=malicious_classes,
    vectorizer=current_vectorizer,
    scaler=current_scaler,
    title="Final Adaptive Model on Final Test",
    verbose=True
)

# 결과 저장
stream_results_df = pd.DataFrame(stream_results)
stream_result_path = os.path.join(RESULT_DIR, "stream_update_results.csv")
stream_results_df.to_csv(stream_result_path, index=False)

final_result_path = os.path.join(RESULT_DIR, "final_test_result.json")
with open(final_result_path, "w", encoding="utf-8") as f:
    json.dump(final_result, f, ensure_ascii=False, indent=2)

print("\nSaved stream results:", stream_result_path)
print("Saved final test result:", final_result_path)

print("\nDone.")

Tokenizer test:
https://secure-login.example.com/account/verify?id=123 => ['tld:com', 'subdomain:secure-login', 'domain:secure-login', 'dpart:secure', 'dpart:login', 'sld:example', 'domain:example', 'dpart:example', 'tldpart:com', 'domain:com', 'dpart:com', 'path:account', 'pseg:account', 'path:verify', 'pseg:verify', 'has_query', 'qkey:id', 'qk:id', 'qv:123', 'tok:https']
http://abc.ru/download/app.apk => ['tld:ru', 'sld:abc', 'domain:abc', 'dpart:abc', 'tldpart:ru', 'domain:ru', 'dpart:ru', 'path:download', 'pseg:download', 'path:app.apk', 'pseg:app', 'pseg:apk', 'ext:apk', 'tok:http', 'tok:abc', 'tok:ru', 'tok:download', 'tok:app', 'tok:apk', 'char:dow']
paypal.com.verify-user-login-security.com/index.php => ['tld:com', 'subdomain:paypal', 'domain:paypal', 'dpart:paypal', 'subdomain:com', 'domain:com', 'dpart:com', 'sld:verify-user-login-security', 'domain:verify-user-login-security', 'dpart:verify', 'dpart:user', 'dpart:login', 'dpart:security', 'tldpart:com', 'domain:com', 'dpart:

In [3]:
# ============================================================
# AI-based Multi-Class Malicious URL Detection
# Adaptive Risk Token Dictionary + Streaming Update
# Safe Full Version + Phishing Confidence Gate
#
# Key idea:
# - Keep the original adaptive model structure.
# - No phishing-specific handcrafted feature boost.
# - No phishing class weight boost.
# - Apply a conservative confidence gate only when the model predicts phishing.
# ============================================================

import os
import re
import json
import math
import urllib.parse
from collections import Counter, deque

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight


# ============================================================
# 0. Config
# ============================================================

SEED = 42
np.random.seed(SEED)

DATA_PATH = "./malicious_phish.csv"

URL_COL = "url"
LABEL_COL = "type"
BENIGN_LABEL = "benign"
PHISHING_LABEL = "phishing"

# Risk dictionary
TOP_K_COMMON_MALICIOUS = 1000
TOP_K_PER_MAL_CLASS = 800
MIN_TOKEN_DF = 5
MIN_MAL_CLASSES_FOR_COMMON = 2
ALPHA = 1.0

# TF-IDF
MAX_TFIDF_FEATURES = 50000
TFIDF_MIN_DF = 3
TFIDF_MAX_DF = 0.95

# Streaming
BATCH_SIZE = 5000
UPDATE_INTERVAL = 3
SLIDING_WINDOW_BATCHES = 6

# Phishing confidence gate
# 모델이 phishing이라고 예측했더라도 확신이 낮으면 2순위 클래스로 돌림.
# precision을 올리는 목적이므로 너무 세게 걸면 phishing recall이 떨어질 수 있음.
USE_PHISHING_GATE_FOR_STREAM = True
USE_PHISHING_GATE_FOR_FINAL = True
PHISHING_MIN_PROB = 0.55
PHISHING_MIN_MARGIN = 0.08

# Output
RISK_DICT_DIR = "./risk_dict_phishing_gate"
RESULT_DIR = "./results_phishing_gate"

os.makedirs(RISK_DICT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 1. Safe URL Tokenizer
# ============================================================

def clean_url_text(url):
    """
    어떤 URL 문자열이 들어와도 urlparse가 죽지 않도록 정리.
    특히 Invalid IPv6 URL을 유발하는 '[' ']' 제거.
    """
    if url is None:
        url = ""

    url = str(url).strip().lower()

    try:
        url = urllib.parse.unquote(url, errors="ignore")
    except Exception:
        pass

    # Invalid IPv6 URL 에러 방지 핵심
    url = url.replace("[", "")
    url = url.replace("]", "")

    # 제어 문자 제거
    url = re.sub(r"[\x00-\x1f\x7f]", "", url)

    return url


def split_by_separators(text):
    """
    URL 내부 문자열을 특수문자 기준으로 분리.
    """
    if text is None:
        return []

    text = clean_url_text(text)

    parts = re.split(r"[^a-zA-Z0-9]+", text)
    return [p for p in parts if len(p) >= 2]


def char_ngrams(token, n_min=3, n_max=4):
    """
    긴 문자열에 대해 character n-gram 생성.
    """
    token = str(token).lower()
    grams = []

    if len(token) < n_min:
        return grams

    for n in range(n_min, n_max + 1):
        if len(token) >= n:
            for i in range(len(token) - n + 1):
                grams.append(f"char:{token[i:i+n]}")

    return grams


def safe_parse_url(url):
    """
    urlparse를 안전하게 수행.
    실패하면 invalid.local로 대체.
    이 함수는 절대 에러를 밖으로 던지지 않음.
    """
    cleaned = clean_url_text(url)

    try:
        parse_target = cleaned

        if not re.match(r"^[a-zA-Z][a-zA-Z0-9+\-.]*://", parse_target):
            parse_target = "http://" + parse_target

        return urllib.parse.urlparse(parse_target)

    except Exception:
        try:
            return urllib.parse.urlparse("http://invalid.local/")
        except Exception:
            return None


def tokenize_url(url):
    """
    URL 구조 기반 보안 특화 tokenizer.
    어떤 URL이 들어와도 절대 예외를 밖으로 던지지 않음.
    """
    try:
        decoded_url = clean_url_text(url)
        parsed = safe_parse_url(decoded_url)

        tokens = []

        if parsed is None:
            general_parts = split_by_separators(decoded_url)
            for p in general_parts:
                tokens.append(f"tok:{p}")
            return tokens if tokens else ["empty_or_invalid_url"]

        host = parsed.netloc.lower() if parsed.netloc else ""
        path = parsed.path.lower() if parsed.path else ""
        query = parsed.query.lower() if parsed.query else ""

        # user:pass@host
        if "@" in host:
            tokens.append("has_userinfo")
            host = host.split("@")[-1]

        # port 제거
        if ":" in host:
            host = host.split(":")[0]

        # www 제거
        if host.startswith("www."):
            host = host[4:]

        # IP host 여부
        if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host):
            tokens.append("host_is_ip")

        # domain / subdomain / tld
        domain_parts = [p for p in host.split(".") if p]

        if domain_parts:
            tld = domain_parts[-1]
            tokens.append(f"tld:{tld}")

            for idx, part in enumerate(domain_parts):
                if idx == len(domain_parts) - 1:
                    tokens.append(f"tldpart:{part}")
                elif idx == len(domain_parts) - 2:
                    tokens.append(f"sld:{part}")
                else:
                    tokens.append(f"subdomain:{part}")

                tokens.append(f"domain:{part}")

                for sub in split_by_separators(part):
                    tokens.append(f"dpart:{sub}")

        # path
        path_segments = [seg for seg in path.split("/") if seg]

        for seg in path_segments:
            tokens.append(f"path:{seg}")

            for p in split_by_separators(seg):
                tokens.append(f"pseg:{p}")

        # extension
        if path_segments:
            last_seg = path_segments[-1]
            if "." in last_seg:
                ext = last_seg.split(".")[-1]
                if 1 <= len(ext) <= 8:
                    tokens.append(f"ext:{ext}")

        # query
        if query:
            tokens.append("has_query")

        try:
            query_pairs = urllib.parse.parse_qsl(query, keep_blank_values=True)
        except Exception:
            query_pairs = []

        for k, v in query_pairs:
            if k:
                tokens.append(f"qkey:{k}")
                for p in split_by_separators(k):
                    tokens.append(f"qk:{p}")

            if v:
                for p in split_by_separators(v):
                    tokens.append(f"qv:{p}")

        # 전체 URL 기준 일반 토큰
        general_parts = split_by_separators(decoded_url)

        for p in general_parts:
            tokens.append(f"tok:{p}")

        # 긴 문자열에 character n-gram 추가
        for p in general_parts:
            if len(p) >= 6:
                tokens.extend(char_ngrams(p, 3, 4))

        # 구조적 패턴 토큰
        if "-" in decoded_url:
            tokens.append("has_hyphen")
        if "_" in decoded_url:
            tokens.append("has_underscore")
        if "%" in decoded_url:
            tokens.append("has_percent_encoding")
        if decoded_url.count(".") >= 4:
            tokens.append("many_dots")
        if len(decoded_url) >= 100:
            tokens.append("very_long_url")

        if not tokens:
            tokens.append("empty_or_invalid_url")

        return tokens

    except Exception:
        return ["tokenizer_error"]


def safe_tokenize(url):
    """
    외부에서 항상 이 tokenizer를 사용.
    """
    try:
        return tokenize_url(url)
    except Exception:
        return ["tokenizer_error"]


# tokenizer 작동 테스트
test_urls = [
    "https://secure-login.example.com/account/verify?id=123",
    "http://abc.ru/download/app.apk",
    "paypal.com.verify-user-login-security.com/index.php",
    "http://[broken-ipv6-url",
    "[invalid]url.com/test"
]

print("Tokenizer test:")
for u in test_urls:
    print(u, "=>", safe_tokenize(u)[:20])


# ============================================================
# 2. Load Dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

print("\nOriginal shape:", df.shape)
print("Columns:", df.columns.tolist())

df = df[[URL_COL, LABEL_COL]].dropna()
df[URL_COL] = df[URL_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(str)

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())


# ============================================================
# 3. Split: Initial Train 40% / Stream Update 30% / Final Test 30%
# ============================================================

# First split:
# 40% -> Initial Train
# 60% -> Temporary pool for Stream Update + Final Test
initial_train_df, temp_df = train_test_split(
    df,
    test_size=0.60,
    stratify=df[LABEL_COL],
    random_state=SEED
)

# Second split:
# temp_df is 60% of total.
# Split it equally:
# 30% -> Stream Update
# 30% -> Final Test
stream_update_df, final_test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[LABEL_COL],
    random_state=SEED
)

print("\nSplit result")
print("Initial Train:", initial_train_df.shape)
print("Stream Update:", stream_update_df.shape)
print("Final Test:", final_test_df.shape)

print("\nInitial Train label distribution")
print(initial_train_df[LABEL_COL].value_counts())

print("\nStream Update label distribution")
print(stream_update_df[LABEL_COL].value_counts())

print("\nFinal Test label distribution")
print(final_test_df[LABEL_COL].value_counts())


# ============================================================
# 4. Label Encoding
# ============================================================

label_encoder = LabelEncoder()

y_initial = label_encoder.fit_transform(initial_train_df[LABEL_COL])
y_stream = label_encoder.transform(stream_update_df[LABEL_COL])
y_final = label_encoder.transform(final_test_df[LABEL_COL])

class_names = list(label_encoder.classes_)
malicious_classes = [c for c in class_names if c != BENIGN_LABEL]
all_class_ids = np.arange(len(class_names))
label_to_id = {label: idx for idx, label in enumerate(class_names)}

benign_id = label_to_id.get(BENIGN_LABEL, None)
phishing_id = label_to_id.get(PHISHING_LABEL, None)

print("\nClass names:", class_names)
print("Malicious classes:", malicious_classes)
print("Benign id:", benign_id)
print("Phishing id:", phishing_id)


# ============================================================
# 5. Adaptive Risk Token Dictionary
# ============================================================

def build_adaptive_risk_dictionary(
    urls,
    labels,
    tokenizer,
    class_names,
    benign_label="benign",
    top_k_common=1000,
    top_k_per_class=800,
    min_df=5,
    min_mal_classes_for_common=2,
    alpha=1.0
):
    """
    위험 토큰 사전 생성.

    common_malicious:
        benign에서는 적게 나오고,
        여러 악성 클래스에서 공통적으로 많이 나오는 토큰.

    class_specific:
        특정 악성 클래스에서 다른 클래스보다 더 강하게 나타나는 토큰.
    """

    malicious_classes = [c for c in class_names if c != benign_label]

    class_token_df = {c: Counter() for c in class_names}
    total_token_df = Counter()
    class_doc_count = Counter()

    for url, label in zip(urls, labels):
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        class_doc_count[label] += 1

        for t in tokens:
            class_token_df[label][t] += 1
            total_token_df[t] += 1

    total_docs = sum(class_doc_count.values())
    benign_docs = class_doc_count[benign_label]
    malicious_docs = total_docs - benign_docs

    # --------------------------------------------------------
    # 5-1. Common malicious tokens
    # --------------------------------------------------------
    common_scores = {}

    for token, total_df in total_token_df.items():
        if total_df < min_df:
            continue

        benign_df = class_token_df[benign_label][token]
        malicious_df = sum(class_token_df[c][token] for c in malicious_classes)

        mal_class_presence = sum(
            1 for c in malicious_classes
            if class_token_df[c][token] > 0
        )

        if mal_class_presence < min_mal_classes_for_common:
            continue

        p_token_mal = (malicious_df + alpha) / (malicious_docs + 2 * alpha)
        p_token_benign = (benign_df + alpha) / (benign_docs + 2 * alpha)

        score = math.log(p_token_mal / p_token_benign)

        if score > 0:
            common_scores[token] = score

    common_malicious = dict(
        sorted(common_scores.items(), key=lambda x: x[1], reverse=True)[:top_k_common]
    )

    # --------------------------------------------------------
    # 5-2. Class-specific malicious tokens
    # --------------------------------------------------------
    class_specific = {c: {} for c in malicious_classes}

    for c in malicious_classes:
        scores = {}

        n_c = class_doc_count[c]
        n_not_c = total_docs - n_c

        for token, total_df in total_token_df.items():
            if total_df < min_df:
                continue

            df_c = class_token_df[c][token]
            df_not_c = total_df - df_c

            p_token_c = (df_c + alpha) / (n_c + 2 * alpha)
            p_token_not_c = (df_not_c + alpha) / (n_not_c + 2 * alpha)

            score = math.log(p_token_c / p_token_not_c)

            if score > 0:
                scores[token] = score

        class_specific[c] = dict(
            sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_per_class]
        )

    return {
        "common_malicious": common_malicious,
        "class_specific": class_specific
    }


def save_risk_dict(risk_dict, version):
    path = os.path.join(RISK_DICT_DIR, f"risk_token_dict_v{version}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(risk_dict, f, ensure_ascii=False, indent=2)
    print(f"Saved risk dictionary: {path}")


def print_risk_dict_preview(risk_dict, top_n=15):
    print("\n===== Common Malicious Risk Tokens =====")
    for token, score in list(risk_dict["common_malicious"].items())[:top_n]:
        print(token, round(score, 4))

    print("\n===== Class-specific Risk Tokens =====")
    for c, d in risk_dict["class_specific"].items():
        print(f"\n[{c}]")
        for token, score in list(d.items())[:top_n]:
            print(token, round(score, 4))


# ============================================================
# 6. Build Initial Risk Dictionary
# ============================================================

print("\nBuilding initial risk dictionary...")

initial_urls = initial_train_df[URL_COL].tolist()
initial_labels = initial_train_df[LABEL_COL].tolist()

risk_dict = build_adaptive_risk_dictionary(
    urls=initial_urls,
    labels=initial_labels,
    tokenizer=safe_tokenize,
    class_names=class_names,
    benign_label=BENIGN_LABEL,
    top_k_common=TOP_K_COMMON_MALICIOUS,
    top_k_per_class=TOP_K_PER_MAL_CLASS,
    min_df=MIN_TOKEN_DF,
    min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
    alpha=ALPHA
)

save_risk_dict(risk_dict, version=0)
print_risk_dict_preview(risk_dict, top_n=20)


# ============================================================
# 7. Feature Engineering
# ============================================================

def risk_score_features(urls, risk_dict, tokenizer, malicious_classes):
    """
    URL 하나를 위험 토큰 사전 기반 feature로 변환.

    feature 구성:
    - common malicious score
    - common malicious match count
    - 각 악성 클래스별 score
    - 각 악성 클래스별 match count
    """
    rows = []

    common_dict = risk_dict["common_malicious"]
    class_dict = risk_dict["class_specific"]

    for url in urls:
        try:
            tokens = set(tokenizer(url))
        except Exception:
            tokens = {"tokenizer_error"}

        row = []

        common_score = 0.0
        common_count = 0

        for t in tokens:
            if t in common_dict:
                common_score += common_dict[t]
                common_count += 1

        row.append(common_score)
        row.append(common_count)

        for c in malicious_classes:
            c_score = 0.0
            c_count = 0

            c_dict = class_dict.get(c, {})

            for t in tokens:
                if t in c_dict:
                    c_score += c_dict[t]
                    c_count += 1

            row.append(c_score)
            row.append(c_count)

        rows.append(row)

    return np.array(rows, dtype=np.float32)


def lexical_features(urls):
    """
    URL 구조 통계 feature.
    """
    rows = []

    for url in urls:
        u = clean_url_text(url)
        length = len(u)

        digit_count = sum(ch.isdigit() for ch in u)
        alpha_count = sum(ch.isalpha() for ch in u)
        special_count = sum(not ch.isalnum() for ch in u)

        dot_count = u.count(".")
        slash_count = u.count("/")
        hyphen_count = u.count("-")
        underscore_count = u.count("_")
        question_count = u.count("?")
        equal_count = u.count("=")
        amp_count = u.count("&")
        percent_count = u.count("%")
        at_count = u.count("@")

        digit_ratio = digit_count / max(length, 1)
        alpha_ratio = alpha_count / max(length, 1)
        special_ratio = special_count / max(length, 1)

        has_ip = 1 if re.search(r"\d{1,3}(\.\d{1,3}){3}", u) else 0
        has_https = 1 if u.startswith("https://") else 0
        has_http = 1 if u.startswith("http://") else 0

        rows.append([
            length,
            digit_count,
            alpha_count,
            special_count,
            dot_count,
            slash_count,
            hyphen_count,
            underscore_count,
            question_count,
            equal_count,
            amp_count,
            percent_count,
            at_count,
            digit_ratio,
            alpha_ratio,
            special_ratio,
            has_ip,
            has_https,
            has_http
        ])

    return np.array(rows, dtype=np.float32)


def build_train_features(urls, risk_dict, malicious_classes):
    """
    Initial train용 feature 생성.
    TF-IDF vectorizer와 scaler는 initial train에 대해서만 fit.
    """
    vectorizer = TfidfVectorizer(
        tokenizer=safe_tokenize,
        token_pattern=None,
        lowercase=False,
        max_features=MAX_TFIDF_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF
    )

    X_tfidf = vectorizer.fit_transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])

    scaler = StandardScaler()
    X_extra_scaled = scaler.fit_transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X, vectorizer, scaler


def build_eval_features(urls, risk_dict, malicious_classes, vectorizer, scaler):
    """
    Stream/Test용 feature 생성.
    vectorizer와 scaler는 initial train에서 fit한 것을 그대로 사용.
    """
    X_tfidf = vectorizer.transform(urls)

    X_risk = risk_score_features(urls, risk_dict, safe_tokenize, malicious_classes)
    X_lex = lexical_features(urls)

    X_extra = np.hstack([X_risk, X_lex])
    X_extra_scaled = scaler.transform(X_extra)

    X = hstack([X_tfidf, csr_matrix(X_extra_scaled)])

    return X


# ============================================================
# 8. Prediction with Phishing Confidence Gate
# ============================================================

def predict_raw(model, X):
    """
    기존 모델 그대로 예측.
    """
    return model.predict(X)


def predict_with_phishing_gate(
    model,
    X,
    phishing_id,
    min_phish_prob=0.55,
    min_margin=0.08
):
    """
    모델이 phishing이라고 예측한 샘플에만 confidence gate 적용.

    동작:
    1. predict_proba로 클래스 확률 계산.
    2. top-1 클래스가 phishing인 경우만 검사.
    3. phishing 확률이 낮거나 top-2와의 margin이 작으면 top-2 클래스로 변경.

    의도:
    - phishing recall을 무리하게 올리는 것이 아니라,
      benign -> phishing 오탐을 줄여 phishing precision을 개선하는 방향.
    """
    if phishing_id is None:
        return model.predict(X)

    try:
        proba = model.predict_proba(X)
    except Exception:
        return model.predict(X)

    pred = np.argmax(proba, axis=1)

    for i in range(len(pred)):
        if pred[i] != phishing_id:
            continue

        sorted_ids = np.argsort(proba[i])[::-1]
        top1 = sorted_ids[0]
        top2 = sorted_ids[1]

        phish_prob = proba[i][phishing_id]
        margin = proba[i][top1] - proba[i][top2]

        if phish_prob < min_phish_prob or margin < min_margin:
            pred[i] = top2

    return pred


def predict_model(
    model,
    X,
    use_phishing_gate=False,
    phishing_id=None,
    min_phish_prob=0.55,
    min_margin=0.08
):
    """
    gate on/off를 선택할 수 있는 통합 예측 함수.
    """
    if use_phishing_gate:
        return predict_with_phishing_gate(
            model=model,
            X=X,
            phishing_id=phishing_id,
            min_phish_prob=min_phish_prob,
            min_margin=min_margin
        )

    return predict_raw(model, X)


# ============================================================
# 9. Initial Model Training
# ============================================================

print("\nBuilding initial features...")

X_initial, vectorizer, scaler = build_train_features(
    urls=initial_train_df[URL_COL].tolist(),
    risk_dict=risk_dict,
    malicious_classes=malicious_classes
)

print("X_initial shape:", X_initial.shape)

model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    n_jobs=-1
)

initial_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_initial
)

model.partial_fit(
    X_initial,
    y_initial,
    classes=all_class_ids,
    sample_weight=initial_sample_weight
)

print("\nInitial model trained.")
print("Phishing confidence gate enabled for stream:", USE_PHISHING_GATE_FOR_STREAM)
print("Phishing confidence gate enabled for final:", USE_PHISHING_GATE_FOR_FINAL)
print("PHISHING_MIN_PROB:", PHISHING_MIN_PROB)
print("PHISHING_MIN_MARGIN:", PHISHING_MIN_MARGIN)


# ============================================================
# 10. Evaluation Function
# ============================================================

def evaluate_model(
    model,
    urls,
    y_true,
    risk_dict,
    malicious_classes,
    vectorizer,
    scaler,
    title="Evaluation",
    verbose=True,
    use_phishing_gate=False,
    phishing_id=None,
    min_phish_prob=0.55,
    min_margin=0.08
):
    X = build_eval_features(
        urls=urls,
        risk_dict=risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=vectorizer,
        scaler=scaler
    )

    y_pred = predict_model(
        model=model,
        X=X,
        use_phishing_gate=use_phishing_gate,
        phishing_id=phishing_id,
        min_phish_prob=min_phish_prob,
        min_margin=min_margin
    )

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    if verbose:
        print(f"\n===== {title} =====")
        print("Phishing gate:", use_phishing_gate)
        if use_phishing_gate:
            print("PHISHING_MIN_PROB:", min_phish_prob)
            print("PHISHING_MIN_MARGIN:", min_margin)
        print("Accuracy:", round(acc, 4))
        print("Macro-F1:", round(macro_f1, 4))
        print("Weighted-F1:", round(weighted_f1, 4))
        print("\nClassification Report")
        print(classification_report(y_true, y_pred, target_names=class_names))
        print("\nConfusion Matrix")
        print(confusion_matrix(y_true, y_pred))

    return {
        "title": title,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "use_phishing_gate": use_phishing_gate,
        "phishing_min_prob": min_phish_prob if use_phishing_gate else None,
        "phishing_min_margin": min_margin if use_phishing_gate else None
    }


# ============================================================
# 11. Stream Utility Functions
# ============================================================

def make_batches(df, batch_size):
    batches = []

    n = len(df)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batches.append(df.iloc[start:end].copy())

    return batches


def summarize_risk_dict_change(old_dict, new_dict, malicious_classes):
    summary = {}

    old_common = set(old_dict["common_malicious"].keys())
    new_common = set(new_dict["common_malicious"].keys())

    summary["common_malicious"] = {
        "added": len(new_common - old_common),
        "removed": len(old_common - new_common),
        "kept": len(old_common & new_common),
        "sample_added": list(new_common - old_common)[:10],
        "sample_removed": list(old_common - new_common)[:10]
    }

    for c in malicious_classes:
        old_tokens = set(old_dict["class_specific"].get(c, {}).keys())
        new_tokens = set(new_dict["class_specific"].get(c, {}).keys())

        summary[c] = {
            "added": len(new_tokens - old_tokens),
            "removed": len(old_tokens - new_tokens),
            "kept": len(old_tokens & new_tokens),
            "sample_added": list(new_tokens - old_tokens)[:10],
            "sample_removed": list(old_tokens - new_tokens)[:10]
        }

    return summary


def print_change_summary(summary):
    print("\nRisk Dictionary Change Summary")

    for key, value in summary.items():
        print(f"\n[{key}]")
        print("added:", value["added"])
        print("removed:", value["removed"])
        print("kept:", value["kept"])
        print("sample_added:", value["sample_added"])
        print("sample_removed:", value["sample_removed"])


# ============================================================
# 12. Streaming Update
# ============================================================

stream_batches = make_batches(stream_update_df, BATCH_SIZE)

print("\nNumber of stream batches:", len(stream_batches))
print("Batch size:", BATCH_SIZE)
print("Update interval:", UPDATE_INTERVAL)
print("Sliding window batches:", SLIDING_WINDOW_BATCHES)

current_model = model
current_risk_dict = risk_dict
current_vectorizer = vectorizer
current_scaler = scaler

recent_stream_buffer = deque(maxlen=SLIDING_WINDOW_BATCHES)

stream_results = []
version = 0

for batch_idx, batch_df in enumerate(stream_batches, start=1):
    print(f"\n\n========== Stream Batch {batch_idx}/{len(stream_batches)} ==========")

    recent_stream_buffer.append(batch_df)

    batch_urls = batch_df[URL_COL].tolist()
    batch_y = label_encoder.transform(batch_df[LABEL_COL])

    # 현재 모델로 stream batch 평가
    # Final Test는 여기서 사용하지 않음.
    batch_result = evaluate_model(
        model=current_model,
        urls=batch_urls,
        y_true=batch_y,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler,
        title=f"Before Update - Stream Batch {batch_idx}",
        verbose=False,
        use_phishing_gate=USE_PHISHING_GATE_FOR_STREAM,
        phishing_id=phishing_id,
        min_phish_prob=PHISHING_MIN_PROB,
        min_margin=PHISHING_MIN_MARGIN
    )

    print(
        f"Before Update Batch {batch_idx} | "
        f"ACC={batch_result['accuracy']:.4f}, "
        f"Macro-F1={batch_result['macro_f1']:.4f}, "
        f"Weighted-F1={batch_result['weighted_f1']:.4f}"
    )

    batch_result["batch_idx"] = batch_idx
    batch_result["dict_version"] = version
    stream_results.append(batch_result)

    # 현재 batch로 classifier incremental update
    # 학습은 기존 방식 그대로 진행. gate는 평가/예측에만 적용.
    X_batch = build_eval_features(
        urls=batch_urls,
        risk_dict=current_risk_dict,
        malicious_classes=malicious_classes,
        vectorizer=current_vectorizer,
        scaler=current_scaler
    )

    batch_sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=batch_y
    )

    current_model.partial_fit(
        X_batch,
        batch_y,
        classes=all_class_ids,
        sample_weight=batch_sample_weight
    )

    # 일정 주기마다 위험 토큰 사전 갱신
    if batch_idx % UPDATE_INTERVAL == 0:
        print(f"\n----- Risk Dictionary Update at Batch {batch_idx} -----")

        version += 1

        recent_df = pd.concat(list(recent_stream_buffer), axis=0)

        # Initial Train + 최근 Stream Window로만 사전 갱신
        # Final Test는 절대 포함하지 않음.
        dict_update_df = pd.concat([initial_train_df, recent_df], axis=0)

        update_urls = dict_update_df[URL_COL].tolist()
        update_labels = dict_update_df[LABEL_COL].tolist()

        new_risk_dict = build_adaptive_risk_dictionary(
            urls=update_urls,
            labels=update_labels,
            tokenizer=safe_tokenize,
            class_names=class_names,
            benign_label=BENIGN_LABEL,
            top_k_common=TOP_K_COMMON_MALICIOUS,
            top_k_per_class=TOP_K_PER_MAL_CLASS,
            min_df=MIN_TOKEN_DF,
            min_mal_classes_for_common=MIN_MAL_CLASSES_FOR_COMMON,
            alpha=ALPHA
        )

        change_summary = summarize_risk_dict_change(
            old_dict=current_risk_dict,
            new_dict=new_risk_dict,
            malicious_classes=malicious_classes
        )

        print_change_summary(change_summary)
        save_risk_dict(new_risk_dict, version=version)

        current_risk_dict = new_risk_dict

        # 새 risk dictionary 기준으로 최근 stream window를 한 번 더 학습
        X_recent = build_eval_features(
            urls=recent_df[URL_COL].tolist(),
            risk_dict=current_risk_dict,
            malicious_classes=malicious_classes,
            vectorizer=current_vectorizer,
            scaler=current_scaler
        )

        y_recent = label_encoder.transform(recent_df[LABEL_COL])

        recent_sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_recent
        )

        current_model.partial_fit(
            X_recent,
            y_recent,
            classes=all_class_ids,
            sample_weight=recent_sample_weight
        )

        print(f"Dictionary version updated to v{version}")


# ============================================================
# 13. Final Test Evaluation
# ============================================================

print("\n\n================================================")
print("Final Test Evaluation")
print("================================================")

final_urls = final_test_df[URL_COL].tolist()

# 13-1. Raw final result: 기존 adaptive 모델 그대로
final_result_raw = evaluate_model(
    model=current_model,
    urls=final_urls,
    y_true=y_final,
    risk_dict=current_risk_dict,
    malicious_classes=malicious_classes,
    vectorizer=current_vectorizer,
    scaler=current_scaler,
    title="Final Adaptive Model on Final Test - RAW",
    verbose=True,
    use_phishing_gate=False,
    phishing_id=phishing_id,
    min_phish_prob=PHISHING_MIN_PROB,
    min_margin=PHISHING_MIN_MARGIN
)

# 13-2. Gated final result: phishing confidence gate 적용
final_result_gate = evaluate_model(
    model=current_model,
    urls=final_urls,
    y_true=y_final,
    risk_dict=current_risk_dict,
    malicious_classes=malicious_classes,
    vectorizer=current_vectorizer,
    scaler=current_scaler,
    title="Final Adaptive Model on Final Test - Phishing Gate",
    verbose=True,
    use_phishing_gate=USE_PHISHING_GATE_FOR_FINAL,
    phishing_id=phishing_id,
    min_phish_prob=PHISHING_MIN_PROB,
    min_margin=PHISHING_MIN_MARGIN
)

# ============================================================
# 14. Save Results
# ============================================================

stream_results_df = pd.DataFrame(stream_results)
stream_result_path = os.path.join(RESULT_DIR, "stream_update_results.csv")
stream_results_df.to_csv(stream_result_path, index=False)

final_result_raw_path = os.path.join(RESULT_DIR, "final_test_result_raw.json")
with open(final_result_raw_path, "w", encoding="utf-8") as f:
    json.dump(final_result_raw, f, ensure_ascii=False, indent=2)

final_result_gate_path = os.path.join(RESULT_DIR, "final_test_result_phishing_gate.json")
with open(final_result_gate_path, "w", encoding="utf-8") as f:
    json.dump(final_result_gate, f, ensure_ascii=False, indent=2)

comparison_result_path = os.path.join(RESULT_DIR, "final_test_comparison_raw_vs_gate.json")
with open(comparison_result_path, "w", encoding="utf-8") as f:
    json.dump({
        "raw": final_result_raw,
        "phishing_gate": final_result_gate,
        "gate_config": {
            "PHISHING_MIN_PROB": PHISHING_MIN_PROB,
            "PHISHING_MIN_MARGIN": PHISHING_MIN_MARGIN,
            "USE_PHISHING_GATE_FOR_STREAM": USE_PHISHING_GATE_FOR_STREAM,
            "USE_PHISHING_GATE_FOR_FINAL": USE_PHISHING_GATE_FOR_FINAL
        }
    }, f, ensure_ascii=False, indent=2)

print("\nSaved stream results:", stream_result_path)
print("Saved raw final test result:", final_result_raw_path)
print("Saved phishing-gate final test result:", final_result_gate_path)
print("Saved comparison result:", comparison_result_path)

print("\nDone.")


Tokenizer test:
https://secure-login.example.com/account/verify?id=123 => ['tld:com', 'subdomain:secure-login', 'domain:secure-login', 'dpart:secure', 'dpart:login', 'sld:example', 'domain:example', 'dpart:example', 'tldpart:com', 'domain:com', 'dpart:com', 'path:account', 'pseg:account', 'path:verify', 'pseg:verify', 'has_query', 'qkey:id', 'qk:id', 'qv:123', 'tok:https']
http://abc.ru/download/app.apk => ['tld:ru', 'sld:abc', 'domain:abc', 'dpart:abc', 'tldpart:ru', 'domain:ru', 'dpart:ru', 'path:download', 'pseg:download', 'path:app.apk', 'pseg:app', 'pseg:apk', 'ext:apk', 'tok:http', 'tok:abc', 'tok:ru', 'tok:download', 'tok:app', 'tok:apk', 'char:dow']
paypal.com.verify-user-login-security.com/index.php => ['tld:com', 'subdomain:paypal', 'domain:paypal', 'dpart:paypal', 'subdomain:com', 'domain:com', 'dpart:com', 'sld:verify-user-login-security', 'domain:verify-user-login-security', 'dpart:verify', 'dpart:user', 'dpart:login', 'dpart:security', 'tldpart:com', 'domain:com', 'dpart:

피싱은 url토큰사전만으로 안되어서 아이디어 덧붙여야함